In [1]:
import pandas as pd

In [ ]:
import pandas as pd
import numpy as np

# ---- 1. Load data ----
abt = pd.read_csv(r'c:\Users\Praphulla\Downloads\Research\raw\Abt-Buy\Abt.csv',encoding='latin1')

buy = pd.read_csv(r'c:\Users\Praphulla\Downloads\Research\raw\Abt-Buy\Buy.csv',encoding='latin1')

mapping = pd.read_csv(r"C:\Users\Praphulla\Downloads\Research\raw\Abt-Buy\abt_buy_perfectMapping.csv",encoding='latin1')

print(abt.shape, buy.shape, mapping.shape)

(1081, 4) (1092, 5) (1097, 2)


In [14]:
# ---- 2. Build positive pairs ----
# Join mapping -> abt (rename to avoid column clashes) -> buy

abt_renamed = abt.rename(columns={'id': 'id_abt', 'name': 'name_abt', 'description':'description_abt', 'price': 'price_abt'})

buy_renamed = buy.rename(columns={'id': 'id_buy', 'name': 'name_buy', 'description':'description_buy', 'price': 'price_buy'})

positives = mapping.rename(columns={'idAbt': 'id_abt', 'idBuy': 'id_buy'})
positives = positives.merge(abt_renamed, on='id_abt', how='left')
positives = positives.merge(buy_renamed, on='id_buy', how='left')
positives['label'] = 1

In [17]:
# ---- 3. Build negative pairs ----
# Set of known true matches, for fast lookup

true_pairs = set(zip(mapping['idAbt'], mapping['idBuy']))

abt_ids = abt['id'].tolist()
buy_ids = buy['id'].tolist()

rng = np.random.default_rng(seed=42)

n_negatives_needed = len(positives)
negative_rows = []
seen_neg_pairs = set()

max_attempts = n_negatives_needed * 20
attempts = 0

while len(negative_rows) < n_negatives_needed and attempts < max_attempts:
    a_id = rng.choice(abt_ids)
    b_id = rng.choice(buy_ids)
    attempts += 1

    if (a_id, b_id) in true_pairs:
        continue
    if(a_id, b_id) in seen_neg_pairs:
        continue

    seen_neg_pairs.add((a_id, b_id))
    negative_rows.append((a_id, b_id))

negatives = pd.DataFrame(negative_rows, columns=['id_abt', 'id_buy'])
negatives = negatives.merge(abt_renamed, on='id_abt', how='left')
negatives = negatives.merge(buy_renamed, on='id_buy', how='left')
negatives['label'] = 0

print("Negative pairs:", negatives.shape)

Negative pairs: (1097, 10)


In [21]:
# ---- 4. Combine, shuffle, sanity-check ----
pairs = pd.concat([positives, negatives], ignore_index=True)
pairs = pairs.sample(frac=1, random_state=42).reset_index(drop=True)

# Keep only the columns Day 2's definition of done asks for
# (you can keep description/price too if you want richer context for later prompts)
pairs_slim = pairs[['id_abt', 'id_buy', 'name_abt', 'name_buy', 'label']]

print(pairs_slim['label'].value_counts())
print(pairs_slim.head(10))

# Eyeball a few of each
print("\n--- Sample positives ---")
print(pairs_slim[pairs_slim.label == 1].sample(5, random_state=1))
print("\n--- Sample negatives ---")
print(pairs_slim[pairs_slim.label == 0].sample(5, random_state=1))

label
1    1097
0    1097
Name: count, dtype: int64
   id_abt     id_buy                                           name_abt  \
0   34959  207383660  Plantronics .Audio 920 Bluetooth Headset - AUD...   
1   33804   90125772  Motorola MotoRokr Portable Bluetooth Car Kit S...   
2   36243  206678505  Monster iCarPlay Wireless 250 FM Transmitter W...   
3   37010  204559210               Onkyo Black Stereo Receiver - TX8255   
4   35477  206359209          Griffin iTrip FM Transmitter - 4052TRPSEB   
5   32664  209975310  iHome Black Clock Radio Audio System For iPod ...   
6   37877  210401451  Apple MacBook Pro 2.4GHz Intel Core 2 Duo Silv...   
7   33938  205144562  Weber Summit E-620 Copper Liquid Propane Gas O...   
8   35010  208117938   LG 30' White Freestanding Gas Range - LRG30357WH   
9   32059  205131849  Unreal Tournament III Video Game For The Sony ...   

                                            name_buy  label  
0  Plantronics .Audio 920 Wireless Earset - 78592-01      1 

In [22]:
# ---- 5. Save ----
import os
os.makedirs('data', exist_ok=True)
pairs_slim.to_csv(r"C:\Users\Praphulla\Downloads\Research\data\abt_buy_pairs.csv", index=False)
print("Saved to data/abt_buy_pairs.csv")

Saved to data/abt_buy_pairs.csv


In [29]:
pairs_slim

,id_abt,id_buy,name_abt,name_buy,label
0,34959,207383660,Plantronics .Audio 920 Bluetooth Headset - AUD...,Plantronics .Audio 920 Wireless Earset - 78592-01,1
1,33804,90125772,Motorola MotoRokr Portable Bluetooth Car Kit S...,Sanus Speaker Mount - WMS3B BLACK,0
2,36243,206678505,Monster iCarPlay Wireless 250 FM Transmitter W...,MONSTER A IP FM-CH 250 iCarPlay Wireless 250 F...,1
3,37010,204559210,Onkyo Black Stereo Receiver - TX8255,Panasonic NNSD797S 1.6 cu. ft. Genius Prestige...,0
4,35477,206359209,Griffin iTrip FM Transmitter - 4052TRPSEB,Griffin iTrip FM Transmitter - 4052-TRPSEB,1
...,...,...,...,...,...
2189,38511,207465595,Audiovox 7' Acrylic Digital Photo Frame - DPF701,Panasonic Lumix DMC-FS3 Digital Camera - Silver,0
2190,31176,205844279,Sony White Cybershot T Series Digital Camera J...,Sony LCJ-THC/B Jacket Case with Stylus - LCJ-T...,1
2191,37856,205520997,Canon Black Leather Case - 3528B001,Bracketron iPod Docking Kit,0
2192,35810,209026638,Canon KP-36IP Color Ink & Paper Set - 7737A001,Toshiba 52RV535U - 52' Widescreen 1080p LCD HD...,0
